# Figure: CombFold on structurally uncharacterized complexes and pLDDT trimming

Self-contained notebook for the thesis figure of sections *Application to structurally uncharacterized complexes* and *Trimming of low-confidence regions in input models*. The comparison of 1 vs 5 input models per pair is in notebook 26.

Panels of the main figure
- **A** Complexes with an assembly and with a confidence above 80, per run (untrimmed, trimmed + interface safeguard, trimmed)
- **B** Fraction of residues removed per pair model
- **C** Best confidence per complex, untrimmed vs trimmed, highlighting complexes that only cross 80 after trimming
- **D** TM-score between trimmed and untrimmed assemblies of the same stoichiometry

Figures are only shown, not saved (uncomment the `savefig` lines if needed).

In [ ]:
import re
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from matplotlib.patches import Patch
from procompa import get_project_root

%config InlineBackend.figure_format = "retina"
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["font.size"] = 8
plt.rcParams["axes.labelsize"] = 9
plt.rcParams["axes.titlesize"] = 9
plt.rcParams["legend.fontsize"] = 7.5

PRJ_ROOT = get_project_root()
DATA_DIR = PRJ_ROOT / "data"
PIPELINE_DIR = DATA_DIR / "Pipeline"

In [ ]:
CONFIDENCE_THRESHOLD = 80
N_PROTEINS_ABOVE = 2              # keep complexes with more than this many distinct proteins
UNCHARACTERIZED_MATCH_CLASSES = ["no_sufficient_match", "no_homology_match"]

RUN_FOLDERS = {
    "untrimmed": "18_CP_model_based_on_pae",
    "safeguard": "20_CP_pae_plddt_trimmed_interface_safeguard",
    "trimmed": "19_CP_pae_plddt_trimmed",
}
RUN_LABELS = {"untrimmed": "Untrimmed", "trimmed": "Trimmed", "safeguard": "Trimmed +\nsafeguard"}
RUN_COLORS = {"untrimmed": "#0b4a8b", "safeguard": "#2a9d8f", "trimmed": "#c0566b"}
NEUTRAL_GREY = "#9e9e9e"

PDB_MATCH_CLASS_CSV = DATA_DIR / "complete_complex_pdb_mapping_v2/all_pdb_matches_with_match_class.csv"
TRIMMING_TM_CSV = PRJ_ROOT / "scripts/Pipeline/effect_of_trimming/effect_of_trimming_on_cf_assembly.csv"
for required_path in [PDB_MATCH_CLASS_CSV, TRIMMING_TM_CSV, *[PIPELINE_DIR / folder for folder in RUN_FOLDERS.values()]]:
    assert required_path.exists(), f"missing input: {required_path}"

## 1. Best CombFold confidence for complexes without a structure

In [ ]:
pdb_match_classes = pl.read_csv(PDB_MATCH_CLASS_CSV)
uncharacterized_complexes = (
    pdb_match_classes
    .filter(pl.col("match_class").is_in(UNCHARACTERIZED_MATCH_CLASSES))
    .filter(pl.col("n_proteins") > N_PROTEINS_ABOVE)
    .select("complex_ac", "match_class")
    .unique()
)
assert uncharacterized_complexes["complex_ac"].is_unique().all(), "a complex has more than one match class"
print(f"{uncharacterized_complexes.height} complexes without a (substantial) structure match")

In [ ]:
CONFIDENCE_COLUMNS = ["CF_1_confidence", "CF_2_confidence", "CF_3_confidence", "CF_true_confidence"]


def best_confidence_expression(column_name: str) -> pl.Expr:
    """'{0: 65.5, 1: 68.0}' -> 68.0 (best assembly of this stoichiometry), '{}' -> null"""
    return (
        pl.col(column_name).cast(pl.String).str.extract_all(r": [\d.]+")
        .list.eval(pl.element().str.slice(2).cast(pl.Float64))
        .list.max()
    )


def load_best_confidence_per_complex(run_name: str) -> pl.DataFrame:
    """Highest confidence over all submitted stoichiometries of one run (null = no assembly)."""
    run_folder = RUN_FOLDERS[run_name]
    results_csv = PIPELINE_DIR / run_folder / f"all_pdb_present_{run_folder}_pool_pipeline_complexes_combfold_results.csv"
    assert results_csv.exists(), f"missing results file: {results_csv}"
    combfold_results = pl.read_csv(results_csv)
    missing_columns = [name for name in CONFIDENCE_COLUMNS if name not in combfold_results.columns]
    assert not missing_columns, f"{results_csv.name}: missing columns {missing_columns}"
    combfold_results = combfold_results.join(uncharacterized_complexes.select("complex_ac"), on="complex_ac", how="inner")
    assert combfold_results["complex_ac"].is_unique().all(), f"{results_csv.name}: more than one row per complex"
    return combfold_results.select(
        "complex_ac",
        pl.max_horizontal(*[best_confidence_expression(name) for name in CONFIDENCE_COLUMNS]).alias(run_name),
    )


best_confidence_by_run = {run_name: load_best_confidence_per_complex(run_name) for run_name in RUN_FOLDERS}

# all runs must contain the same complexes, otherwise the comparison is not paired
submitted_complexes = set(best_confidence_by_run["untrimmed"]["complex_ac"])
for run_name, run_confidence in best_confidence_by_run.items():
    assert set(run_confidence["complex_ac"]) == submitted_complexes, f"{run_name}: other complexes than the untrimmed run"

uncharacterized_confidence = uncharacterized_complexes.filter(pl.col("complex_ac").is_in(list(submitted_complexes)))
for run_name, run_confidence in best_confidence_by_run.items():
    uncharacterized_confidence = uncharacterized_confidence.join(run_confidence, on="complex_ac", how="inner")
assert uncharacterized_confidence.height == len(submitted_complexes), "join lost complexes"
print(f"{uncharacterized_complexes.height} uncharacterized complexes, {uncharacterized_confidence.height} submitted to CombFold")

In [ ]:
run_summary = pl.DataFrame([
    {
        "run": run_name,
        "n_submitted": uncharacterized_confidence.height,
        "n_with_output": uncharacterized_confidence[run_name].is_not_null().sum(),
        "n_above_threshold": uncharacterized_confidence.filter(pl.col(run_name) > CONFIDENCE_THRESHOLD).height,
    }
    for run_name in RUN_FOLDERS
])
print(run_summary)

# complexes that only pass the threshold after trimming
crossed_by_trimming = uncharacterized_confidence.filter(
    (pl.col("untrimmed") <= CONFIDENCE_THRESHOLD) & (pl.col("trimmed") > CONFIDENCE_THRESHOLD)
)
lost_by_trimming = uncharacterized_confidence.filter(
    (pl.col("untrimmed") > CONFIDENCE_THRESHOLD) & ~(pl.col("trimmed") > CONFIDENCE_THRESHOLD).fill_null(False)
)
above_only_trimmed_without_untrimmed_output = uncharacterized_confidence.filter(
    pl.col("untrimmed").is_null() & (pl.col("trimmed") > CONFIDENCE_THRESHOLD)
)
print(f"\n{crossed_by_trimming.height} complexes above {CONFIDENCE_THRESHOLD} only after trimming, "
      f"untrimmed confidence {crossed_by_trimming['untrimmed'].min():.1f} to {crossed_by_trimming['untrimmed'].max():.1f}")
print(f"{lost_by_trimming.height} complexes above {CONFIDENCE_THRESHOLD} untrimmed but not trimmed")
print(f"{above_only_trimmed_without_untrimmed_output.height} complexes above {CONFIDENCE_THRESHOLD} trimmed without an untrimmed assembly")
print(crossed_by_trimming.sort("untrimmed").select("complex_ac", *[pl.col(name).round(1) for name in RUN_FOLDERS]))

## 2. Fraction of residues removed per pair model

In [ ]:
TRIMMED_RUN_DIR = PIPELINE_DIR / RUN_FOLDERS["trimmed"] / "CombFold"
SAFEGUARD_RUN_DIR = PIPELINE_DIR / RUN_FOLDERS["safeguard"] / "CombFold"
CA_LINE_PATTERN = re.compile(rb"^ATOM.{8} CA [ A]", re.MULTILINE)


def count_ca_atoms_regex(pdb_path: Path) -> int:
    with open(pdb_path, "rb") as pdb_handle:
        return len(CA_LINE_PATTERN.findall(pdb_handle.read()))


def count_ca_atoms_fast(pdb_path: Path) -> int:
    with open(pdb_path, "rb") as pdb_handle:
        return pdb_handle.read().count(b" CA ")


def find_pair_models(run_dir: Path, model_subdir: str) -> dict:
    """One pdb path per unique pair (first occurrence across pools)."""
    pair_model_paths = {}
    for pool_dir in sorted(run_dir.glob("*_pool_input")):
        for pdb_path in (pool_dir / model_subdir).glob("AFM_*_unrelaxed_rank_1_*.pdb"):
            pair_name = pdb_path.name.removeprefix("AFM_").split("_unrelaxed")[0]
            pair_model_paths.setdefault(pair_name, pdb_path)
    assert pair_model_paths, f"no pair models in {run_dir}/*_pool_input/{model_subdir}"
    return pair_model_paths


pair_models_by_input = {
    "untrimmed": find_pair_models(TRIMMED_RUN_DIR, "pdbs_untrimmed"),
    "trimmed": find_pair_models(TRIMMED_RUN_DIR, "pdbs"),
    "safeguard": find_pair_models(SAFEGUARD_RUN_DIR, "pdbs"),
}
print({input_name: len(paths) for input_name, paths in pair_models_by_input.items()})

# the fast count must equal the strict regex count
for pair_model_paths in pair_models_by_input.values():
    for pdb_path in list(pair_model_paths.values())[:20]:
        assert count_ca_atoms_fast(pdb_path) == count_ca_atoms_regex(pdb_path), f"CA count mismatch in {pdb_path}"

In [ ]:
count_jobs = [
    (input_name, pair_name, pdb_path)
    for input_name, pair_model_paths in pair_models_by_input.items()
    for pair_name, pdb_path in pair_model_paths.items()
]
with ThreadPoolExecutor(max_workers=64) as executor:
    ca_counts = list(executor.map(count_ca_atoms_fast, (job[2] for job in count_jobs)))

pair_model_lengths = (
    pl.DataFrame({
        "input": [job[0] for job in count_jobs],
        "pair": [job[1] for job in count_jobs],
        "n_residues": ca_counts,
    })
    .pivot(on="input", index="pair", values="n_residues")
)

pairs_in_all_inputs = set.intersection(*[set(paths) for paths in pair_models_by_input.values()])
n_dropped_pairs = pair_model_lengths.height - len(pairs_in_all_inputs)
pair_model_lengths = pair_model_lengths.drop_nulls(subset=list(pair_models_by_input))
assert pair_model_lengths.height == len(pairs_in_all_inputs), "rows left != pairs present in all inputs"
print(f"dropped {n_dropped_pairs} pairs missing in at least one input, {pair_model_lengths.height} left")

assert (pair_model_lengths["trimmed"] <= pair_model_lengths["untrimmed"]).all(), "trimmed longer than untrimmed"
assert (pair_model_lengths["safeguard"] <= pair_model_lengths["untrimmed"]).all(), "safeguard longer than untrimmed"

pair_model_lengths = pair_model_lengths.with_columns(
    removed_percent_trimmed=100 * (pl.col("untrimmed") - pl.col("trimmed")) / pl.col("untrimmed"),
    removed_percent_safeguard=100 * (pl.col("untrimmed") - pl.col("safeguard")) / pl.col("untrimmed"),
)
removed_percent_trimmed = pair_model_lengths["removed_percent_trimmed"].to_numpy()
removed_percent_safeguard = pair_model_lengths["removed_percent_safeguard"].to_numpy()
print(f"median removed per pair, trimmed {np.median(removed_percent_trimmed):.1f}%, "
      f"safeguard {np.median(removed_percent_safeguard):.1f}%")

## 3. TM-score between trimmed and untrimmed assemblies

In [ ]:
trimming_tm_scores = pl.read_csv(TRIMMING_TM_CSV)
print(trimming_tm_scores.columns)
for required_column in ["n_proteins", "tm_score_norm_by_trimmed"]:
    assert required_column in trimming_tm_scores.columns, f"missing column {required_column}"

trimming_tm_with_score = trimming_tm_scores.filter(pl.col("tm_score_norm_by_trimmed").is_not_null())
assert not trimming_tm_with_score["tm_score_norm_by_trimmed"].is_nan().any(), "NaN TM-scores"
assert (trimming_tm_with_score["n_proteins"] > N_PROTEINS_ABOVE).all(), "complexes with too few proteins"
print(f"{trimming_tm_scores.height - trimming_tm_with_score.height} rows without TM-score, "
      f"{trimming_tm_with_score.height} plotted, median TM-score {trimming_tm_with_score['tm_score_norm_by_trimmed'].median():.2f}")

tm_trimmed_vs_untrimmed = trimming_tm_with_score["tm_score_norm_by_trimmed"].to_numpy()
# for the text, how the low TM-scores distribute over complex size
print(trimming_tm_with_score.filter(pl.col("tm_score_norm_by_trimmed") < 0.5)["n_proteins"].value_counts().sort("n_proteins"))

## 4. Main figure

In [ ]:
def style_axis(axis):
    axis.spines[["top", "right"]].set_visible(False)


def add_panel_letter(axis, letter, x_offset=-0.2):
    axis.text(x_offset, 1.08, letter, transform=axis.transAxes, fontsize=12, fontweight="bold", va="bottom")


figure, axes_grid = plt.subplots(2, 2, figsize=(7.5, 6.6), gridspec_kw={"hspace": 0.5, "wspace": 0.4})
axis_counts, axis_removed, axis_paired, axis_tm = axes_grid.flat

# A, complexes with an assembly and with a confidence above the threshold
n_submitted = uncharacterized_confidence.height
for bar_position, run_name in enumerate(RUN_FOLDERS):
    summary_row = run_summary.filter(pl.col("run") == run_name).row(0, named=True)
    n_above = summary_row["n_above_threshold"]
    n_below = summary_row["n_with_output"] - n_above
    axis_counts.bar(bar_position, n_above, width=0.65, color=RUN_COLORS[run_name])
    axis_counts.bar(bar_position, n_below, bottom=n_above, width=0.65, color=RUN_COLORS[run_name], alpha=0.35)
    axis_counts.text(bar_position, n_above / 2, str(n_above), ha="center", va="center",
                     fontsize=8, fontweight="bold", color="white")
    axis_counts.text(bar_position, n_above + n_below + 2, str(n_above + n_below), ha="center", va="bottom", fontsize=8)
axis_counts.set_xticks(range(len(RUN_FOLDERS)), [RUN_LABELS[run_name] for run_name in RUN_FOLDERS])
axis_counts.set_ylim(0, n_submitted * 1.35)
axis_counts.set_ylabel("Complexes with assembly")
axis_counts.set_title(f"Complexes without structure (n = {n_submitted})")
axis_counts.legend(
    handles=[Patch(facecolor="#555555", label=f"Confidence > {CONFIDENCE_THRESHOLD}"),
             Patch(facecolor="#555555", alpha=0.35, label=f"Confidence ≤ {CONFIDENCE_THRESHOLD}")],
    loc="upper center", frameon=False, ncol=2, handlelength=1, columnspacing=1,
)

# C, untrimmed vs trimmed confidence per complex
paired_confidence = uncharacterized_confidence.drop_nulls(subset=["untrimmed", "trimmed"])
confidence_untrimmed = paired_confidence["untrimmed"].to_numpy()
confidence_trimmed = paired_confidence["trimmed"].to_numpy()
crossed_mask = (confidence_untrimmed <= CONFIDENCE_THRESHOLD) & (confidence_trimmed > CONFIDENCE_THRESHOLD)
confidence_limits = (
    np.floor(min(confidence_untrimmed.min(), confidence_trimmed.min()) / 5) * 5,
    np.ceil(max(confidence_untrimmed.max(), confidence_trimmed.max()) / 5) * 5,
)
axis_paired.plot(confidence_limits, confidence_limits, color=NEUTRAL_GREY, linestyle="--", linewidth=0.8, zorder=0)
axis_paired.axvline(CONFIDENCE_THRESHOLD, color=NEUTRAL_GREY, linestyle=":", linewidth=0.8, zorder=0)
axis_paired.axhline(CONFIDENCE_THRESHOLD, color=NEUTRAL_GREY, linestyle=":", linewidth=0.8, zorder=0)
axis_paired.scatter(confidence_untrimmed[~crossed_mask], confidence_trimmed[~crossed_mask],
                    s=10, color=RUN_COLORS["untrimmed"], alpha=0.6, edgecolors="none",
                    label=f"Other ({(~crossed_mask).sum()})")
axis_paired.scatter(confidence_untrimmed[crossed_mask], confidence_trimmed[crossed_mask],
                    s=16, color=RUN_COLORS["trimmed"], edgecolors="k", linewidths=0.3,
                    label=f"> {CONFIDENCE_THRESHOLD} only after trimming ({crossed_mask.sum()})")
axis_paired.set_xlim(confidence_limits)
axis_paired.set_ylim(confidence_limits)
axis_paired.set_aspect("equal")
axis_paired.set_xlabel("CF confidence, untrimmed")
axis_paired.set_ylabel("CF confidence, trimmed")
axis_paired.legend(loc="lower right", frameon=False, handletextpad=0.3)

# B, residues removed per pair model
removed_bins = np.linspace(0, 100, 41)
for removed_percent, input_name, input_label in [
    (removed_percent_safeguard, "safeguard", "Trimmed + safeguard"),
    (removed_percent_trimmed, "trimmed", "Trimmed"),
]:
    median_removed = np.median(removed_percent)
    axis_removed.hist(removed_percent, bins=removed_bins, color=RUN_COLORS[input_name], alpha=0.35)
    axis_removed.hist(removed_percent, bins=removed_bins, color=RUN_COLORS[input_name], histtype="step", linewidth=1.2)
    axis_removed.axvline(median_removed, color=RUN_COLORS[input_name], linestyle="--", linewidth=1)
    axis_removed.plot([], [], color=RUN_COLORS[input_name], linewidth=4, alpha=0.6,
                      label=f"{input_label} (median {median_removed:.1f}%)")
axis_removed.set_xlim(0, 100)
axis_removed.set_xlabel("Residues removed per pair model (%)")
axis_removed.set_ylabel("Pair models")
axis_removed.legend(frameon=False, loc="upper right")

# D, TM-score between trimmed and untrimmed assemblies
tm_bins = np.arange(0, 1.0001, 0.05)
axis_tm.hist(tm_trimmed_vs_untrimmed, bins=tm_bins, color=RUN_COLORS["untrimmed"], edgecolor="white", linewidth=0.4)
median_tm = np.median(tm_trimmed_vs_untrimmed)
axis_tm.axvline(median_tm, color="k", linestyle="--", linewidth=1, label=f"Median {median_tm:.2f} (n = {len(tm_trimmed_vs_untrimmed)})")
axis_tm.set_xlim(0, 1)
axis_tm.set_xlabel("TM-score, trimmed vs untrimmed assembly")
axis_tm.set_ylabel("Assemblies")
axis_tm.legend(frameon=False, loc="upper left")

for axis, panel_letter in zip(axes_grid.flat, "ABCD"):
    style_axis(axis)
    add_panel_letter(axis, panel_letter)

plt.show()
# figure.savefig("cf_uncharacterized_trimming.pdf", bbox_inches="tight")

## 5. Numbers quoted in the text

In [ ]:
summary_by_run = {row["run"]: row for row in run_summary.iter_rows(named=True)}
print(f"submitted to CF                         {n_submitted}")
for run_name in RUN_FOLDERS:
    print(f"{run_name:<10} assemblies {summary_by_run[run_name]['n_with_output']:>4}, "
          f"confidence > {CONFIDENCE_THRESHOLD} {summary_by_run[run_name]['n_above_threshold']:>3}")
print(f"above {CONFIDENCE_THRESHOLD} only after trimming          {crossed_by_trimming.height}")
print(f"median residues removed, trimmed        {np.median(removed_percent_trimmed):.1f}%")
print(f"median residues removed, safeguard      {np.median(removed_percent_safeguard):.1f}%")
print(f"median TM-score trimmed vs untrimmed    {median_tm:.2f} (n = {len(tm_trimmed_vs_untrimmed)})")